# 04. Premapping Finetune + NL-to-JOILang Full Pipeline

논문(Service Pre-mapping) 프로토콜의 전체 파이프라인을 한 노트북에서 단계별로 실행하고 검증합니다.
각 단계는 독립적으로 실행 결과를 눈으로 확인할 수 있도록 중간 산출물을 출력합니다.

```text
파이프라인 (논문 매핑)
 1. 데이터 로드            : datasets/service_list_ver2.0.1.json + JOICommands-280.csv
 2. 역할 문서 코퍼스       : VALUE/FUNCTION 분리 문서 + 디바이스 별칭 (Sec. Pre-mapping)
 3. 쿼리 풀               : 템플릿 / 자연어(LLM 역할 감수) / 절 풀 + answerability 필터
 4. 학습쌍               : (query, positive role doc) — 예시 명령은 문서에 미포함(누수 차단)
 5. e5-small fine-tune    : MultipleNegativesRankingLoss (epochs 10, batch 32, lr 2e-5)
 6. 검색 평가             : base vs fine-tuned, nDCG@10 / Recall@5 / Recall@10
 7. Service Pre-mapping   : Eq.(1) hybrid w=0.6, VALUE/FUNCTION split + 절 분해
 8. gpt_mg 검색 번들      : fine-tuned 인코더로 디바이스 번들 재생성 (w=0.6)
 9. NL-to-JOILang 생성    : gpt_mg/version0_15_update20260413 파이프라인 + gpt-4.1-mini
10. strict DET 평가       : utils/det_evaluator.strict_det_evaluate_row vs gt
```

사용하는 저장소 파일:

```text
JOILang-Server/datasets/service_list_ver2.0.1.json
JOILang-Server/datasets/JOICommands-280.csv
JOILang-Server/gpt_mg/version0_15_update20260413/utils/{pipeline_common,retrieval_context,local_llm_client}.py
JOILang-Server/gpt_mg/version0_13/  (legacy monolith 프롬프트 자산)
JOILang-Server/utils/det_evaluator.py
ModelMagementApp/datasets/service_list_ver2.0.1_retrieval_eng.json   (읽기 전용 참조)
ModelMagementApp/embedding/embedding_result_v5_ver2.0.1/             (읽기 전용 참조)
```

중요 구분:

- GA 시리즈 노트북(00–03)은 저장소 코드를 subprocess로만 구동하지만, 이 노트북은 fine-tuning과 검색을
  커널 안에서 직접 수행합니다. 커널을 `joi` conda 환경(torch + sentence-transformers)으로 실행하세요.
- API 키는 절대 하드코딩하지 않습니다. 셸 환경변수 `OPENAI_API_KEY`를 상속받아
  `JOI_V15_OPENAI_API_KEY`로 매핑합니다. 자연어 역할 라벨은 `tutorials/data/query_roles_ver2.0.1.json`
  캐시를 우선 사용하므로 캐시가 있으면 라벨링 API 호출이 없습니다.
- `ModelMagementApp`은 읽기 전용 참조입니다. 이 노트북은 그 안의 어떤 파일도 수정하지 않습니다.
- 생성/평가 단계는 gpt-4.1-mini 호출 비용이 발생합니다. 행 수는 `N_DET_ROWS`(기본 8)로 제한됩니다.


## 1. Setup

경로·시드·출력 디렉토리를 설정합니다. 모든 노브는 환경변수로 재정의할 수 있고 기본값이 들어 있습니다.
산출물은 `artifacts/premapping_pipeline_runs/<RUN_TAG>/` 아래에 저장됩니다.

In [1]:
# ============================================================
# Setup
# ============================================================
import csv, json, os, random, re, sys, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch

BASE_DIR = Path(os.environ.get("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server"))
MMA_DIR = Path(os.environ.get("MODELMAGEMENT_DIR", "/home/mgjeong/Desktop/llm/ModelMagementApp"))  # read-only
V15_DIR = BASE_DIR / "gpt_mg" / "version0_15_update20260413"
TUT_DIR = BASE_DIR / "tutorials"

RUN_TAG = os.environ.get("RUN_TAG", "v201_paper")
ART_DIR = BASE_DIR / "artifacts" / "premapping_pipeline_runs" / RUN_TAG
CKPT_DIR = ART_DIR / "models" / "e5-small-split-v201"   # dir name keeps "e5" so e5 query prefixing stays active
BUNDLE_DIR = ART_DIR / "retrieval_bundle_e5_ft"

SEED = int(os.environ.get("SEED", "2025"))
GEN_MODEL = os.environ.get("GEN_MODEL", "gpt-4.1-mini")
OPENAI_ENDPOINT = os.environ.get("JOI_V15_OPENAI_ENDPOINT", "https://api.openai.com/v1/chat/completions")
RETRAIN_FINETUNE = os.environ.get("RETRAIN_FINETUNE", "false").lower() == "true"
N_DET_ROWS = int(os.environ.get("N_DET_ROWS", "8"))
RUN_SERVER_ENTRY = os.environ.get("RUN_SERVER_ENTRY", "false").lower() == "true"

for p in (BASE_DIR, MMA_DIR, V15_DIR):
    assert p.exists(), f"missing: {p}"
ART_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| device:", DEVICE)
print("artifacts:", ART_DIR)

python: 3.10.20 | torch: 2.6.0+cu124 | device: cuda
artifacts: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/premapping_pipeline_runs/v201_paper


## 2. Data load

canonical 카탈로그(`service_list_ver2.0.1.json`)를 로드하고, 검색·프롬프트에서 쓰는
VALUE/FUNCTION 평탄화 목록을 노트북 안에서 직접 유도합니다(파생 과정이 투명하도록).
자연어 예시 코퍼스(`service_list_ver2.0.1_retrieval_eng.json`)와 최종 테스트셋
`JOICommands-280.csv`도 함께 로드합니다. 280의 gt는 소문자 `{device}_{service}` 호출 규약을 씁니다.

In [2]:
# ============================================================
# Load catalog / natural-example corpus / JOICommands-280
# ============================================================
CATALOG_PATH = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"
RETRIEVAL_JSON = MMA_DIR / "datasets" / "service_list_ver2.0.1_retrieval_eng.json"
DATASET_280 = BASE_DIR / "datasets" / "JOICommands-280.csv"

catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
catalog_value, catalog_function = [], []
for device, services in catalog.items():
    for service, meta in services.items():
        entry = {"device": device, "service": service, **meta}
        (catalog_value if meta.get("type") == "value" else catalog_function).append(entry)

natural_catalog = json.loads(RETRIEVAL_JSON.read_text(encoding="utf-8"))
with DATASET_280.open(encoding="utf-8-sig") as f:   # utf-8-sig: strip BOM on the index column
    rows280 = list(csv.DictReader(f))

n_examples = sum(len(v.get("examples", [])) for v in natural_catalog.values())
print(f"catalog: {len(catalog)} devices, {len(catalog_value)} VALUE / {len(catalog_function)} FUNCTION services")
print(f"natural corpus: {len(natural_catalog)} devices, {n_examples} example commands")
print(f"JOICommands-280: {len(rows280)} rows, columns = {list(rows280[0].keys())}")
print("gt sample (row 1):", json.loads(rows280[0]["gt"])["script"])

catalog: 50 devices, 114 VALUE / 81 FUNCTION services
natural corpus: 50 devices, 1372 example commands
JOICommands-280: 280 rows, columns = ['index', 'category', 'command_kor', 'command_eng', 'connected_devices', 'gt']
gt sample (row 1): (#Dishwasher).dishwasher_setdishwashermode("dry")


## 3. Role-document corpus

디바이스마다 VALUE 문서와 FUNCTION 문서를 만듭니다. 문서는 서비스 설명만 담고
(예시 명령은 절대 넣지 않아 학습/평가 누수가 원천 차단됩니다), 구어체 명칭이 흔한
디바이스에는 별칭(aliases)을 넣습니다. 이 역할 문서가 fine-tuning의 positive 문서이자
검색 평가의 코퍼스입니다.

In [3]:
# ============================================================
# Role documents (VALUE / FUNCTION per device) + aliases
# ============================================================
CAMEL_RE = re.compile(r"(?<!^)(?=[A-Z])")

def humanize(name):
    return re.sub(r"\s+", " ", CAMEL_RE.sub(" ", str(name).replace("_", " "))).strip()

# ver2.0.1 device names (RobotCleaner->RobotVacuumCleaner, SmartPlug->Plug, Curtain/Blind->WindowCovering)
ALIASES = {
    "AirConditioner": ["cooler", "aircon", "AC", "air conditioning"],
    "Television": ["TV"],
    "Light": ["lamp", "bulb"],
    "RobotVacuumCleaner": ["vacuum", "robot vacuum", "robot cleaner"],
    "Plug": ["outlet", "smart plug"],
    "EmailProvider": ["email", "mail"],
    "Speaker": ["music player"],
    "WindowCovering": ["curtain", "blinds", "drapes", "shade"],
    "DoorLock": ["door lock"],
    "WeatherProvider": ["forecast"],
}

by_dev = {}
for role, cat in (("VALUE", catalog_value), ("FUNCTION", catalog_function)):
    for e in cat:
        by_dev.setdefault(e["device"], {"VALUE": [], "FUNCTION": []})[role].append(e)
DEVICES = sorted(by_dev)

def service_line(e):
    return f"{humanize(e['service'])}: {str(e.get('descriptor') or '').strip()}"

def alias_line(dev):
    return [f"Also called: {', '.join(ALIASES[dev])}."] if dev in ALIASES else []

role_docs, merged_docs = {}, {}
for dev, roles in by_dev.items():
    d = humanize(dev)
    for role in ("VALUE", "FUNCTION"):
        if roles[role]:
            role_docs[f"{dev}::{role}"] = "\n".join(
                [f"{d} ({dev}) {role} services."] + alias_line(dev) + [service_line(e) for e in roles[role]])
    merged_docs[dev] = "\n".join(
        [f"{d} ({dev}) services."] + alias_line(dev)
        + [service_line(e) for e in roles["VALUE"] + roles["FUNCTION"]])

print(f"corpus: {len(role_docs)} role docs / {len(merged_docs)} merged docs (descriptions only)")
print("\n--- sample role doc: Dishwasher::FUNCTION ---")
print(role_docs["Dishwasher::FUNCTION"])

corpus: 80 role docs / 50 merged docs (descriptions only)

--- sample role doc: Dishwasher::FUNCTION ---
Dishwasher (Dishwasher) FUNCTION services.
Set Dishwasher Mode: Allows for the control of the dishwasher mode


## 4. Query pools

세 종류의 학습용 쿼리 풀을 만듭니다.

- 템플릿 싱글: 카탈로그 메타데이터에서 결정적으로 생성(역할이 구성상 확정). 10%/10%는 val/test로 보류.
- 자연어 싱글: retrieval 코퍼스의 예시 명령 1,372건. 각 문장의 역할(condition→VALUE / action→FUNCTION /
  fragment→제외)은 gpt-4.1-mini가 감수하며, 결과는 `tutorials/data/query_roles_ver2.0.1.json`에 캐시됩니다.
  이후 answerability 필터(gold 디바이스를 식별할 변별력 있는 단어가 문장에 존재, 문서 빈도 df ≤ 4)를 통과해야 하고,
  소스 편중(AirConditioner 424/1,372) 때문에 디바이스당 상한 70건을 둡니다.
- 절 풀: 조건절(VALUE)·행동절(FUNCTION)을 디바이스별로 생성해 절반은 학습, 절반은 보류.

In [4]:
# ============================================================
# Query pools: template / natural (LLM-audited) / clause
# ============================================================
ROLES_CACHE = TUT_DIR / "data" / "query_roles_ver2.0.1.json"
MAX_EX_PER_SERVICE = 6
HELDOUT_RATIO = 0.1
NATURAL_CAP_PER_DEVICE = 70
LABEL_MODEL = "gpt-4.1-mini"

def dedupe(items):
    seen, out = set(), []
    for it in items:
        k = re.sub(r"\s+", " ", str(it)).strip().lower()
        if k and k not in seen:
            seen.add(k); out.append(re.sub(r"\s+", " ", str(it)).strip())
    return out

def parse_enums(enum_desc_list):
    vals = []
    for item in enum_desc_list or []:
        c = str(item).replace("\u2022", "").strip()
        v = (c.split(" - ", 1)[0] if " - " in c else c.split()[0] if c.split() else "").strip()
        if v and v.lower() not in {x.lower() for x in vals}:
            vals.append(v)
    return vals

def template_value_examples(device, service, meta):
    d, s = humanize(device), humanize(service).lower()
    ret = str(meta.get("return_descriptor") or s).strip()
    ex = [f"What is the current {s} of the {d}?", f"Check {d} {s}",
          f"Tell me the {s} status of the {d}", f"Get {d} {s} info",
          f"What is the {ret.lower()} of the {d}?"]
    for v in parse_enums(meta.get("enums_descriptor"))[:4]:
        ex += [f"If the {d} {s} is {v}, send a notification",
               f"When the {d} {s} becomes {v}, alert me",
               f"Wait until the {d} {s} is {v}"]
    return ex

def template_function_examples(device, service, meta):
    d, s = humanize(device), humanize(service)
    arg_t = str(meta.get("argument_type") or "").upper()
    arg_d = str(meta.get("argument_descriptor") or s).strip().lower()
    enums = parse_enums(meta.get("enums_descriptor"))
    ex = []
    if arg_t.startswith("ENUM") and enums:
        for v in enums[:6]:
            ex += [f"Set the {d} {arg_d} to {v}", f"Change the {d} to {v}", f"Switch the {d} to {v} mode"]
    elif "DOUBLE" in arg_t or "INTEGER" in arg_t:
        for v in ("20", "50", "80"):
            ex += [f"Set the {d} {arg_d} to {v}", f"Adjust the {d} {arg_d} to {v}"]
    elif "STRING" in arg_t or "BOOL" in arg_t or "BINARY" in arg_t:
        ex += [f"Set the {d} {arg_d} to hello", f"Update the {d} {arg_d}"]
    else:
        sl = s.lower()
        ex += [f"{sl} the {d.lower()}", f"Please {sl} for the {d.lower()}",
               f"Use the {d} to {sl}", f"Trigger {sl} for the {d}"]
    return ex

def clause_lists(dev):
    d, dl = humanize(dev), humanize(dev).lower()
    c_list, a_list = [], []
    for e in by_dev[dev]["VALUE"]:
        s = humanize(e["service"]).lower()
        enums = parse_enums(e.get("enums_descriptor"))
        rtype = str(e.get("return_type") or "").upper()
        if enums:
            c_list += [f"If the {d} {s} is {enums[0]}", f"If the {dl} is in {enums[0]} mode"]
            if len(enums) > 1:
                c_list += [f"When the {d} {s} becomes {enums[1]}", f"When the {dl} switches to {enums[1]}"]
        elif "DOUBLE" in rtype or "INTEGER" in rtype:
            c_list += [f"If the {d} {s} is above 25", f"When the {dl} reading goes above 25"]
        else:
            c_list += [f"When the {d} {s} changes", f"Once the {dl} state changes"]
    for e in by_dev[dev]["FUNCTION"]:
        s = humanize(e["service"])
        arg_t = str(e.get("argument_type") or "").upper()
        arg_d = str(e.get("argument_descriptor") or s).strip().lower()
        enums = parse_enums(e.get("enums_descriptor"))
        if arg_t.startswith("ENUM") and enums:
            a_list += [f"set the {d} {arg_d} to {enums[0]}", f"switch the {dl} to {enums[0]}"]
            if len(enums) > 1:
                a_list += [f"put the {dl} in {enums[1]} mode"]
        elif "DOUBLE" in arg_t or "INTEGER" in arg_t:
            a_list += [f"set the {d} {arg_d} to 50", f"adjust the {dl} to 20"]
        else:
            a_list += [f"{s.lower()} the {dl}", f"run {s.lower()} on the {dl}"]
    return dedupe(c_list), dedupe(a_list)

# --- (1) template singles: 80/10/10 ---
seen, tmpl_singles = set(), []
for role, cat in (("VALUE", catalog_value), ("FUNCTION", catalog_function)):
    gen = template_value_examples if role == "VALUE" else template_function_examples
    for e in cat:
        for q in dedupe(gen(e["device"], e["service"], e))[:MAX_EX_PER_SERVICE]:
            if q.lower() not in seen:
                seen.add(q.lower())
                tmpl_singles.append({"query": q, "device": e["device"], "role": role})
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(tmpl_singles))
n_held = int(round(len(tmpl_singles) * HELDOUT_RATIO))
tmpl_test = [tmpl_singles[int(i)] for i in idx[:n_held]]
tmpl_val = [tmpl_singles[int(i)] for i in idx[n_held:2 * n_held]]
tmpl_train = [tmpl_singles[int(i)] for i in idx[2 * n_held:]]

# --- (2) natural singles: LLM-audited roles (cached) + answerability + device cap ---
natural_items = [(dev, str(q).strip()) for dev, obj in natural_catalog.items() if dev in by_dev
                 for q in obj.get("examples", []) or [] if str(q).strip()]

if ROLES_CACHE.exists():
    labels = json.loads(ROLES_CACHE.read_text(encoding="utf-8"))["labels"]
    print(f"role labels: cache hit ({len(labels)} entries) -> no API call")
else:
    from openai import OpenAI
    api_key = os.getenv("OPENAI_API_KEY")
    assert api_key, "OPENAI_API_KEY is required to build the role-label cache"
    client = OpenAI(api_key=api_key)
    PROMPT = (
        "Classify each numbered smart-home phrase into exactly one class:\n"
        '- "condition": states or asks about a device state/status (could follow "if")\n'
        '- "action": an imperative command that actuates a device\n'
        '- "fragment": incomplete, truncated, or not classifiable\n\n'
        'Answer with a JSON object mapping each phrase NUMBER (as a string) to its label:\n'
        '{"labels": {"1": "condition", "2": "action", ...}}\n\nPhrases:\n')
    labels, valid = {}, {"condition", "action", "fragment"}
    for start in range(0, len(natural_items), 50):
        chunk = natural_items[start:start + 50]
        numbered = "\n".join(f"{i + 1}. {q}" for i, (_, q) in enumerate(chunk))
        resp = client.chat.completions.create(model=LABEL_MODEL, temperature=0,
                                              response_format={"type": "json_object"},
                                              messages=[{"role": "user", "content": PROMPT + numbered}])
        out = json.loads(resp.choices[0].message.content)["labels"]
        for i, (dev, q) in enumerate(chunk):
            lab = out.get(str(i + 1), "fragment")
            labels[f"{dev}\t{q}"] = lab if lab in valid else "fragment"
        print(f"  labeled {start + len(chunk)}/{len(natural_items)}", flush=True)
    ROLES_CACHE.parent.mkdir(parents=True, exist_ok=True)
    ROLES_CACHE.write_text(json.dumps({"model": LABEL_MODEL, "labels": labels}, ensure_ascii=False, indent=1),
                           encoding="utf-8")

ROLE_OF = {"condition": "VALUE", "action": "FUNCTION"}
nat_singles, n_frag = [], 0
for dev, q in natural_items:
    lab = labels.get(f"{dev}\t{q}", "fragment")
    if lab == "fragment":
        n_frag += 1
        continue
    nat_singles.append({"query": q, "device": dev, "role": ROLE_OF[lab]})

# answerability: the query must carry a DISCRIMINATIVE content word of its gold device (df <= 4)
STOP = set("the a an is are was in on off to of for and or if when only it this that be at with as by then now me my please state".split())
REF = {dev: set(re.findall(r"[a-z]+", (humanize(dev) + " " + " ".join(ALIASES.get(dev, [])) + " "
                                       + merged_docs[dev]).lower())) for dev in DEVICES}
DF = Counter(t for dev in DEVICES for t in REF[dev])

def answerable(q, dev):
    toks = set(re.findall(r"[a-z]+", q.lower())) - STOP
    return any(t in REF[dev] and DF[t] <= 4 for t in toks)

n_before = len(nat_singles)
nat_singles = [q for q in nat_singles if answerable(q["query"], q["device"])]
n_uncapped = len(nat_singles)

by_device_pool = {}
for q in nat_singles:
    by_device_pool.setdefault(q["device"], []).append(q)
capped = []
for dev in sorted(by_device_pool):
    items = by_device_pool[dev]
    if len(items) > NATURAL_CAP_PER_DEVICE:
        keep = sorted(rng.choice(len(items), size=NATURAL_CAP_PER_DEVICE, replace=False).tolist())
        items = [items[i] for i in keep]
    capped.extend(items)
nat_singles = capped

nidx = rng.permutation(len(nat_singles))
n_nheld = int(round(len(nat_singles) * HELDOUT_RATIO))
nat_test = [nat_singles[int(i)] for i in nidx[:n_nheld]]
nat_val = [nat_singles[int(i)] for i in nidx[n_nheld:2 * n_nheld]]
nat_train = [nat_singles[int(i)] for i in nidx[2 * n_nheld:]]

# --- (3) clause pools: half train / half held out ---
crng = random.Random(SEED)
clause_train = []
for dev in DEVICES:
    c_list, a_list = clause_lists(dev)
    for role, lst in (("VALUE", c_list), ("FUNCTION", a_list)):
        lst = list(lst); crng.shuffle(lst)
        half = (len(lst) + 1) // 2
        clause_train += [{"query": q, "device": dev, "role": role} for q in lst[half:]]

print(f"template singles: {len(tmpl_singles)} -> train {len(tmpl_train)} / val {len(tmpl_val)} / test {len(tmpl_test)}")
print(f"natural singles : {len(natural_items)} raw, {n_frag} fragments, {n_before - n_uncapped} unanswerable, "
      f"{n_uncapped - len(nat_singles)} over device cap -> train {len(nat_train)} / val {len(nat_val)} / test {len(nat_test)}")
print(f"clause pool (train halves): {len(clause_train)}")

role labels: cache hit (1372 entries) -> no API call
template singles: 919 -> train 735 / val 92 / test 92
natural singles : 1372 raw, 177 fragments, 12 unanswerable, 299 over device cap -> train 708 / val 88 / test 88
clause pool (train halves): 191


## 5. Fine-tuning pairs

학습 쿼리(템플릿 train + 자연어 train + 절 train)를 각자의 gold 디바이스 역할 문서와 짝지어
(query, positive_doc) 쌍을 만듭니다. 디바이스에 해당 역할 버킷이 없으면 반대 역할 문서로
대체하고, 풀 간 정확히 중복되는 쌍은 한 번만 기록합니다. e5의 `query: `/`passage: ` 프리픽스는
저장하지 않고 학습 시에 부가합니다(원본 레시피와 동일).

In [5]:
# ============================================================
# Build (query, positive role doc) pairs
# ============================================================
ft_queries = tmpl_train + nat_train + clause_train
pairs, written, n_dup, n_fallback = [], set(), 0, 0
for it in ft_queries:
    key = (it["query"].lower(), it["device"], it["role"])
    if key in written:
        n_dup += 1
        continue
    written.add(key)
    doc_key = f"{it['device']}::{it['role']}"
    if doc_key not in role_docs:
        doc_key = f"{it['device']}::{'FUNCTION' if it['role'] == 'VALUE' else 'VALUE'}"
        n_fallback += 1
    doc = role_docs.get(doc_key)
    if doc:
        pairs.append({"query": it["query"], "device": it["device"], "role": it["role"],
                      "path": "split", "positive_doc": doc})

PAIRS_PATH = ART_DIR / "finetune_pairs_split.jsonl"
with PAIRS_PATH.open("w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

print(f"pairs: {len(pairs)} written ({len(tmpl_train)} template + {len(nat_train)} natural + "
      f"{len(clause_train)} clause, {n_dup} cross-pool duplicates dropped, {n_fallback} role fallbacks)")
print("saved ->", PAIRS_PATH)
for p in (pairs[0], pairs[-1]):
    print(f"\n[{p['role']:8s}] {p['query']}")
    print("   doc:", p["positive_doc"].splitlines()[0])

pairs: 1554 written (735 template + 708 natural + 191 clause, 80 cross-pool duplicates dropped, 0 role fallbacks)
saved -> /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/premapping_pipeline_runs/v201_paper/finetune_pairs_split.jsonl

[VALUE   ] What is the current spin speed of the laundry dryer of the Laundry Dryer?
   doc: Laundry Dryer (LaundryDryer) VALUE services.

[FUNCTION] stop the window covering
   doc: Window Covering (WindowCovering) FUNCTION services.


## 6. Fine-tune `intfloat/e5-small` (MultipleNegativesRankingLoss)

원본 레시피 그대로: MNRL, epochs 10, batch 32, lr 2e-5, warmup 10%, max_seq_length 256,
`query: `/`passage: ` 프리픽스. 배치 내 다른 쌍의 문서가 in-batch negative로 쓰입니다.
체크포인트가 이미 있으면 재사용합니다(`RETRAIN_FINETUNE=true`로 강제 재학습).

In [6]:
# ============================================================
# MNRL fine-tuning of intfloat/e5-small on the split pairs
# ============================================================
from torch.utils.data import DataLoader
from sentence_transformers import InputExample, SentenceTransformer, losses

BASE_ENCODER = "intfloat/e5-small"
EPOCHS, BATCH, LR, WARMUP_RATIO = 10, 32, 2e-5, 0.1

if CKPT_DIR.exists() and not RETRAIN_FINETUNE:
    print("reusing existing checkpoint:", CKPT_DIR)
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    examples = [InputExample(texts=["query: " + p["query"], "passage: " + p["positive_doc"]]) for p in pairs]
    model = SentenceTransformer(BASE_ENCODER).float()
    model.max_seq_length = min(model.max_seq_length or 256, 256)
    loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
    loss = losses.MultipleNegativesRankingLoss(model)
    steps = len(loader) * EPOCHS
    t0 = time.perf_counter()
    model.fit(train_objectives=[(loader, loss)], epochs=EPOCHS,
              warmup_steps=max(10, int(WARMUP_RATIO * steps)),
              optimizer_params={"lr": LR}, output_path=str(CKPT_DIR), show_progress_bar=False)
    print(f"fine-tuned {len(examples)} pairs, {EPOCHS} epochs ({steps} steps) "
          f"in {time.perf_counter() - t0:.0f}s -> {CKPT_DIR}")
    del model; torch.cuda.empty_cache()

/home/mgjeong/miniconda3/envs/joi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/tmp/ipykernel_557314/1769185573.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4689.06it/s]

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


...

{'train_runtime': '70.55', 'train_samples_per_second': '220.3', 'train_steps_per_second': '3.544', 'train_loss': '0.8554', 'epoch': '10'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.60it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.58it/s]

fine-tuned 1554 pairs, 10 epochs (490 steps) in 71s -> /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/premapping_pipeline_runs/v201_paper/models/e5-small-split-v201


## 7. Retrieval evaluation (held-out queries)

학습에 쓰지 않은 보류 쿼리(템플릿 val+test, 자연어 val+test)로 base와 fine-tuned 인코더를
같은 프로토콜로 비교합니다: 역할 문서 80건 전체를 hybrid 점수
`0.6 · minmax(dense) + 0.4 · sparse` (sparse = 0.7·minmax(BM25) + 0.3·minmax(lexical))로 랭킹한 뒤
디바이스 단위로 접어 gold 디바이스에 대한 nDCG@10 / Recall@5 / Recall@10을 계산합니다.

In [7]:
# ============================================================
# Held-out retrieval evaluation: base vs fine-tuned
# ============================================================
import math

def _tokenize(text):
    return re.findall(r"[0-9a-zA-Z_]+", (text or "").lower())

class SimpleBM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [_tokenize(t) for t in texts]
        self.doc_lens = np.array([len(d) for d in self.docs], dtype=np.float32)
        self.avgdl = float(self.doc_lens.mean()) if len(self.doc_lens) else 1.0
        self.tf, df = [], {}
        for doc in self.docs:
            c = {}
            for tok in doc:
                c[tok] = c.get(tok, 0) + 1
            self.tf.append(c)
            for tok in c:
                df[tok] = df.get(tok, 0) + 1
        n = max(1, len(self.docs))
        self.idf = {t: math.log(1.0 + (n - v + 0.5) / (v + 0.5)) for t, v in df.items()}

    def get_scores(self, query):
        q = _tokenize(query)
        out = np.zeros(len(self.docs), dtype=np.float32)
        for i, c in enumerate(self.tf):
            dl = self.doc_lens[i]
            denom = self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))
            out[i] = sum(self.idf.get(t, 0.0) * (c[t] * (self.k1 + 1.0)) / (c[t] + denom)
                         for t in q if t in c)
        return out

def _minmax(x):
    if x.size == 0:
        return x
    den = float(x.max() - x.min())
    return np.zeros_like(x) if abs(den) < 1e-9 else (x - x.min()) / den

DENSE_W = 0.6   # paper operating point w
DOC_KEYS = sorted(role_docs)
DOC_TEXTS = [role_docs[k] for k in DOC_KEYS]
BM25 = SimpleBM25(DOC_TEXTS)
TOKSETS = [set(_tokenize(t)) for t in DOC_TEXTS]

def sparse_scores(query):
    qt = set(_tokenize(query)); qn = max(1, len(qt))
    lex = np.array([len(qt & ts) / math.sqrt(max(1, qn * max(1, len(ts)))) for ts in TOKSETS], dtype=np.float32)
    return 0.7 * _minmax(BM25.get_scores(query)) + 0.3 * _minmax(lex)

def ndcg_at_k(ranked, gold, k=10):
    dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(ranked[:k]) if d == gold)
    return dcg  # single gold -> idcg = 1

def eval_arm(model_path, queries):
    m = SentenceTransformer(model_path, device=DEVICE)
    doc_embs = np.asarray(m.encode(["passage: " + t for t in DOC_TEXTS],
                                   normalize_embeddings=True, show_progress_bar=False))
    ndcgs, r5, r10, lats = [], [], [], []
    for q in queries:
        t0 = time.perf_counter()
        q_emb = np.asarray(m.encode(["query: " + q["query"]], normalize_embeddings=True,
                                    show_progress_bar=False))
        final = DENSE_W * _minmax((q_emb @ doc_embs.T).flatten()) + (1 - DENSE_W) * sparse_scores(q["query"])
        devs, seen_d = [], set()
        for i in np.argsort(-final):
            dev = DOC_KEYS[int(i)].split("::")[0]
            if dev not in seen_d:
                seen_d.add(dev); devs.append(dev)
            if len(devs) >= 10:
                break
        lats.append((time.perf_counter() - t0) * 1000)
        ndcgs.append(ndcg_at_k(devs, q["device"], 10))
        r5.append(float(q["device"] in devs[:5]))
        r10.append(float(q["device"] in devs[:10]))
    del m; torch.cuda.empty_cache()
    return {"nDCG@10": float(np.mean(ndcgs)), "Recall@5": float(np.mean(r5)),
            "Recall@10": float(np.mean(r10)), "latency_ms": float(np.mean(lats))}

heldout = tmpl_val + tmpl_test + nat_val + nat_test
print(f"held-out queries: {len(heldout)} (template {len(tmpl_val) + len(tmpl_test)}, "
      f"natural {len(nat_val) + len(nat_test)})\n")
results = {}
for name, path in (("e5-small (base)", BASE_ENCODER), ("e5-small (fine-tuned)", str(CKPT_DIR))):
    results[name] = eval_arm(path, heldout)
    r = results[name]
    print(f"{name:24s} nDCG@10={r['nDCG@10']:.4f} R@5={r['Recall@5']:.4f} "
          f"R@10={r['Recall@10']:.4f} {r['latency_ms']:.1f}ms")
assert results["e5-small (fine-tuned)"]["nDCG@10"] >= results["e5-small (base)"]["nDCG@10"], \
    "fine-tuned encoder should not be worse than base on held-out queries"

held-out queries: 360 (template 184, natural 176)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3677.92it/s]

e5-small (base)          nDCG@10=0.9969 R@5=1.0000 R@10=1.0000 8.4ms


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4920.11it/s]

e5-small (fine-tuned)    nDCG@10=0.9990 R@5=1.0000 R@10=1.0000 7.5ms


## 8. Service Pre-mapping runtime (paper Eq. (1), split + 절 분해)

런타임 pre-mapping은 서비스 단위 shortlist를 만듭니다. 질의를 규칙 기반으로
조건절(→ VALUE 버킷)과 행동절(→ FUNCTION 버킷)로 분해하고(알림 상용구는 행동으로 치지 않음),
각 버킷을 역할 프리픽스(`condition: `/`action: `)가 붙은 dense 질의 + BM25로 검색해
역할당 상위 6개, 병합 후 총 12개의 shortlist를 만듭니다. JOICommands-280의 실제 질의로 확인합니다.

In [8]:
# ============================================================
# Split pre-mapping with clause decomposition (service level)
# ============================================================
_COND_CLAUSE_RE = re.compile(
    r"^(if|when|whenever|while|until|once|as soon as|the moment|in case|suppose)\b[^,]*", re.IGNORECASE)
_NOTIFY_RE = re.compile(
    r"^(please )?(send (me )?an? ?(notification|alert|message)|notify( me)?|alert( me)?|let me know)\b",
    re.IGNORECASE)

def decompose_query(query):
    m = _COND_CLAUSE_RE.match(query.strip())
    if m and "," in query:
        condition = m.group(0).strip()
        action = query[query.index(",") + 1:].strip()
        if condition and action and not _NOTIFY_RE.match(action):
            return condition, action
    return query, query

def entry_text(e, kind):
    extra = (str(e.get("return_descriptor", "")) if kind == "value"
             else " ".join([str(e.get("argument_descriptor", "")), str(e.get("argument_type", "")),
                            str(e.get("argument_bounds", ""))]))
    return f"{e['device']} {e['service']} {str(e.get('descriptor', ''))} {extra}"

ROLE_ENC_PREFIX = {"value": "condition: ", "function": "action: "}
TOP_K_PER_ROLE, MAX_TOTAL = 6, 12

ft_model = SentenceTransformer(str(CKPT_DIR), device=DEVICE)

class Bucket:
    def __init__(self, entries, kind):
        self.entries, self.kind = entries, kind
        self.texts = [entry_text(e, kind) for e in entries]
        self.embs = np.asarray(ft_model.encode(["passage: " + t for t in self.texts],
                                               normalize_embeddings=True, show_progress_bar=False))
        self.bm25 = SimpleBM25(self.texts)
        self.toksets = [set(_tokenize(t)) for t in self.texts]

    def search(self, clause, top_k):
        q = "query: " + ROLE_ENC_PREFIX[self.kind] + clause
        q_emb = np.asarray(ft_model.encode([q], normalize_embeddings=True, show_progress_bar=False))
        dense = (q_emb @ self.embs.T).flatten()
        qt = set(_tokenize(clause)); qn = max(1, len(qt))
        lex = np.array([len(qt & ts) / math.sqrt(max(1, qn * max(1, len(ts)))) for ts in self.toksets],
                       dtype=np.float32)
        sparse = 0.7 * _minmax(self.bm25.get_scores(clause)) + 0.3 * _minmax(lex)
        final = DENSE_W * _minmax(dense) + (1 - DENSE_W) * sparse
        order = np.argsort(-final)[:top_k]
        return [(self.entries[int(i)], float(final[int(i)]), self.kind) for i in order]

value_bucket = Bucket(catalog_value, "value")
function_bucket = Bucket(catalog_function, "function")

def premap_split(query, top_k_per_role=TOP_K_PER_ROLE, max_total=MAX_TOTAL):
    q_value, q_function = decompose_query(query)
    hits = value_bucket.search(q_value, top_k_per_role) + function_bucket.search(q_function, top_k_per_role)
    hits.sort(key=lambda h: h[1], reverse=True)
    return (q_value, q_function), hits[:max_total]

demo_queries = [rows280[0]["command_eng"]]
demo_queries += [r["command_eng"] for r in rows280
                 if r["command_eng"].split(",")[0].lower().startswith(("if ", "when "))][:2]
for q in demo_queries:
    (qv, qf), shortlist = premap_split(q)
    print("=" * 78)
    print("Q:", q)
    if (qv, qf) != (q, q):
        print(f"   decomposed -> VALUE: {qv!r} | FUNCTION: {qf!r}")
    for e, s, kind in shortlist:
        print(f"   {s:.4f} {kind:8s} {e['device']}.{e['service']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3834.02it/s]

Q: Switch the dishwasher to dry mode.
   1.0000 value    Dishwasher.DishwasherMode
   1.0000 function Dishwasher.SetDishwasherMode
   0.5755 value    LaundryDryer.LaundryDryerMode
   0.5622 function LaundryDryer.SetLaundryDryerMode
   0.5051 function LaundryDryer.SetSpinSpeed
   0.4862 value    Dehumidifier.DehumidifierMode
   0.4787 function RobotVacuumCleaner.SetRobotVacuumCleanerModeMode
   0.4725 function Humidifier.SetHumidifierMode
   0.4645 value    LaundryDryer.SpinSpeed
   0.4495 function Dehumidifier.SetDehumidifierMode
   0.4369 value    Humidifier.HumidifierMode
   0.4218 value    Oven.OvenMode
Q: If the audio recorder is recording, stop recording and save as test.mp3; otherwise, start recording.
   decomposed -> VALUE: 'If the audio recorder is recording' | FUNCTION: 'stop recording and save as test.mp3; otherwise, start recording.'
   1.0000 function AudioRecorder.RecordStop
   0.9883 value    AudioRecorder.RecordStatus
   0.9436 value    AudioRecorder.AudioFile
   0.8853

## 9. gpt_mg retrieval bundle 재생성 (fine-tuned 인코더)

gpt_mg의 `version0_15_update20260413` 파이프라인은 디바이스 수준 pre-mapping을
`(metadata.json, hybrid_config.json, doc_embs_e5.npy)` 번들로 수행합니다. 스톡 번들의 문서
텍스트(디바이스 50건)는 그대로 두고, 문서 임베딩만 방금 fine-tuning한 e5-small로 재인코딩한
커스텀 번들을 만듭니다. dense 가중치는 논문 운영점 `w = 0.6`으로 설정합니다.
번들 생성 후 파이프라인의 `HybridRetriever`로 in-process 검색을 검증합니다.

In [9]:
# ============================================================
# Build a fine-tuned retrieval bundle for the gpt_mg pipeline
# ============================================================
sys.path.insert(0, str(V15_DIR))   # its modules import as "utils.*" relative to this dir
from utils.retrieval_context import encode_texts_dense, load_retrieval_config, HybridRetriever

STOCK_BUNDLE = MMA_DIR / "embedding" / "embedding_result_v5_ver2.0.1"
stock_meta = json.loads((STOCK_BUNDLE / "metadata.json").read_text(encoding="utf-8"))
keys, texts = stock_meta["keys"], stock_meta["texts"]

doc_embs = encode_texts_dense(str(CKPT_DIR), ["passage: " + t for t in texts],
                              batch_size=16, max_length=512, device=DEVICE)
doc_embs = doc_embs / np.linalg.norm(doc_embs, axis=1, keepdims=True)

BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
np.save(BUNDLE_DIR / "doc_embs_e5.npy", doc_embs.astype(np.float32))
(BUNDLE_DIR / "metadata.json").write_text(json.dumps({
    "model_id": str(CKPT_DIR), "model_id_dense": str(CKPT_DIR),
    "source_json": str(RETRIEVAL_JSON), "retrieval_json": str(RETRIEVAL_JSON),
    "num_docs": len(keys), "dim": int(doc_embs.shape[1]),
    "keys": keys, "texts": texts, "saved_at_unix": time.time(),
}, ensure_ascii=False, indent=1), encoding="utf-8")
(BUNDLE_DIR / "hybrid_config.json").write_text(json.dumps({
    "version": 1, "model_id_dense": str(CKPT_DIR), "bm25_pickled": False,
    "best_A": 20, "best_B": 20, "best_w": [0.6, 0.4],   # paper operating point w = 0.6
}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"bundle: {doc_embs.shape[0]} docs x {doc_embs.shape[1]} dims -> {BUNDLE_DIR}")

cfg = load_retrieval_config({
    "service_context_mode": "retrieval_fallback",
    "retrieval_topk": 10, "retrieval_mode": "hybrid",
    "retrieval_json_path": str(RETRIEVAL_JSON),
    "retrieval_bundle_dir": str(BUNDLE_DIR),
    "retrieval_model_dir": str(CKPT_DIR),
    "retrieval_device": "cpu",
})
hits = HybridRetriever(cfg).search("When it rains, close the window covering.", topk=10)
print("sanity search:", [(h["rank"], h["device"], round(float(h.get("score", 0.0)), 4)) for h in hits])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4210.34it/s]

bundle: 50 docs x 384 dims -> /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/premapping_pipeline_runs/v201_paper/retrieval_bundle_e5_ft


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4751.76it/s]

sanity search: [(1, 'WindowCovering', 5.9338), (2, 'Door', 1.8973), (3, 'Valve', 1.8444), (4, 'Television', 0.1452), (5, 'LaundryDryer', 0.1439), (6, 'WeatherProvider', 0.1398), (7, 'RainSensor', 0.1365), (8, 'Speaker', 0.1363), (9, 'Humidifier', 0.1329), (10, 'CloudServiceProvider', 0.1267)]


## 10. NL-to-JOILang 생성 (gpt_mg version0_15_update20260413 + gpt-4.1-mini)

gpt_mg 파이프라인 그대로: `build_prompt_values()`가 pre-mapping shortlist를 프롬프트 값에 심고
(`service_context_mode="retrieval_fallback"`, 방금 만든 fine-tuned 번들 사용),
`render_prompt_bundle()`이 legacy v13 monolith 프롬프트를 렌더링하고,
`local_llm_client.call(mode="openai")`이 gpt-4.1-mini를 호출합니다.

규칙: `connected_devices`가 비어 있지 않으면 retrieval은 생략되고 연결 디바이스가 shortlist가
됩니다(pre-mapping은 디바이스 문맥이 없는 질의에 적용). 두 경우 모두 확인합니다.
검색 워커는 이 커널과 같은 파이썬(`JOI_V15_RETRIEVAL_PYTHON`)을 사용합니다.

In [10]:
# ============================================================
# gpt_mg pipeline: premap -> prompt render -> gpt-4.1-mini
# ============================================================
os.environ.setdefault("JOI_V15_OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
assert os.environ["JOI_V15_OPENAI_API_KEY"], "export OPENAI_API_KEY before starting the kernel"
os.environ["JOI_V15_RETRIEVAL_PYTHON"] = sys.executable   # persistent retrieval worker python

from utils.pipeline_common import SERVICE_SCHEMA_DEFAULT, load_service_schema, load_genome, \
    build_prompt_values, render_prompt_bundle
from utils.local_llm_client import call as llm_call

schema = load_service_schema(SERVICE_SCHEMA_DEFAULT)          # datasets/service_list_ver2.0.1.json
genome = load_genome(V15_DIR / "genomes" / "example_genome.json")
PROMPT_ASSETS = BASE_DIR / "gpt_mg" / "version0_13"

def parse_joilang_json(text):
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(text).strip(), flags=re.S)
    m = re.search(r"\{.*\}", t, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def generate_one(row, row_no=0):
    values = build_prompt_values(
        row_no, row, schema, candidate_strategy="direct",
        service_context_mode="retrieval_fallback",
        retrieval_topk=10, retrieval_mode="hybrid",
        retrieval_json_path=str(RETRIEVAL_JSON),
        retrieval_bundle_dir=str(BUNDLE_DIR),
        retrieval_model_dir=str(CKPT_DIR),
        retrieval_device="cpu",
    )
    system, user, manifest = render_prompt_bundle(
        genome, values=values, command_text=values["command_text"], phase="generate",
        prompt_render_mode="legacy_v13_monolith", prompt_assets_dir=str(PROMPT_ASSETS))
    t0 = time.perf_counter()
    resp = llm_call(system, user, mode="openai", model=GEN_MODEL, endpoint=OPENAI_ENDPOINT,
                    temperature=0.0, max_tokens=1024)
    latency = time.perf_counter() - t0
    return {"values": values, "system": system, "user": user, "resp": resp,
            "parsed": parse_joilang_json(resp.get("content", "")), "latency_sec": latency}

# --- demo A: no connected devices -> retrieval pre-mapping is used ---
demo_row = rows280[0]
out = generate_one(demo_row, 1)
snip = json.loads(out["values"]["service_list_snippet"])
print("command:", demo_row["command_eng"])
print("snippet_source:", snip["snippet_source"], "| retrieval status:", snip["retrieval"]["status"])
print("premap shortlist:", [h["category"] for h in snip["retrieval"].get("shortlist", [])])
print(f"prompt size: system {len(out['system'])} chars / user {len(out['user'])} chars "
      f"| latency {out['latency_sec']:.1f}s")
print("\ngenerated JOILang:", json.dumps(out["parsed"], ensure_ascii=False, indent=1))
print("\ngt              :", demo_row["gt"])

prompts_dir = ART_DIR / "prompts"; prompts_dir.mkdir(exist_ok=True)
(prompts_dir / "demo_row1_prompt.txt").write_text(
    "=== SYSTEM ===\n" + out["system"] + "\n\n=== USER ===\n" + out["user"], encoding="utf-8")
print("\nprompt snapshot ->", prompts_dir / "demo_row1_prompt.txt")

# --- demo B: connected devices present -> retrieval skipped (shortlist = connected devices) ---
cd_row = next(r for r in rows280 if str(r.get("connected_devices", "")).strip())
cd_values = build_prompt_values(0, cd_row, schema, candidate_strategy="direct",
                                service_context_mode="retrieval_fallback",
                                retrieval_json_path=str(RETRIEVAL_JSON),
                                retrieval_bundle_dir=str(BUNDLE_DIR),
                                retrieval_model_dir=str(CKPT_DIR), retrieval_device="cpu")
cd_snip = json.loads(cd_values["service_list_snippet"])
print("\n[connected-devices row]", cd_row["command_eng"][:60])
print("snippet_source:", cd_snip["snippet_source"], "| retrieval status:", cd_snip["retrieval"]["status"])

command: Switch the dishwasher to dry mode.
snippet_source: service_retrieval_fallback | retrieval status: used
premap shortlist: ['Dishwasher', 'ArmRobot', 'LaundryDryer', 'Switch', 'RobotVacuumCleaner', 'Light', 'Dehumidifier', 'Oven', 'AirPurifier', 'RiceCooker']
prompt size: system 107326 chars / user 87 chars | latency 3.2s

generated JOILang: {
 "name": "SetDishwasherToDryMode",
 "cron": "",
 "period": -1,
 "code": "(#Dishwasher).dishwasher_setdishwashermode(\"dry\")"
}

gt              : {"name": "", 
"cron": "", 
"period": 0, 
"script": "(#Dishwasher).dishwasher_setdishwashermode(\"dry\")"}

prompt snapshot -> /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/premapping_pipeline_runs/v201_paper/prompts/demo_row1_prompt.txt

[connected-devices row] Set the living room light to yellow.
snippet_source: connected_devices_only | retrieval status: skipped_connected_devices_present


## 11. Batch generation + strict DET 평가

카테고리 1–8에서 행을 라운드로빈으로 `N_DET_ROWS`개 뽑아 생성하고,
서버의 공식 평가기 `utils/det_evaluator.strict_det_evaluate_row`로 gt와 비교합니다.

해석 주의: `det_pass`(임계 70)는 코드 유사도·스케줄·서비스 커버리지 등의 가중 종합 점수입니다.
호출명까지 엄밀히 맞는지는 `gt_exact`·`failure_reasons`로 확인하세요.
(`utils` 패키지 이름이 version0_15와 겹치므로 평가기는 파일 경로로 직접 로드합니다.)

관찰: 현재 v13 monolith 프롬프트는 일회성 명령에 `"period": -1`을 지시하지만 JOICommands-280 gt는 `"period": 0`을 사용합니다. 그래서 코드가 정확히 일치해도 `period_mismatch`가 남아 `gt_exact`가 False가 되고 det_score가 약 7.5점 깎입니다. 프롬프트 자산의 period 규약을 gt와 통일하면 사라지는 항목입니다.

In [11]:
# ============================================================
# Batch: generate N rows -> strict DET evaluation vs gt
# ============================================================
import importlib.util
import pandas as pd

spec = importlib.util.spec_from_file_location("det_evaluator", BASE_DIR / "utils" / "det_evaluator.py")
det_evaluator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(det_evaluator)
strict_det_evaluate_row = det_evaluator.strict_det_evaluate_row

by_cat = {}
for i, r in enumerate(rows280):
    by_cat.setdefault(r["category"], []).append((i, r))
picked, depth = [], 0
while len(picked) < N_DET_ROWS and depth < max(len(v) for v in by_cat.values()):
    for cat in sorted(by_cat, key=int):
        if depth < len(by_cat[cat]) and len(picked) < N_DET_ROWS:
            picked.append(by_cat[cat][depth])
    depth += 1

records, gen_dump = [], []
for i, row in picked:
    out = generate_one(row, i + 1)
    snip = json.loads(out["values"]["service_list_snippet"])
    candidate = out["parsed"] if out["parsed"] is not None else str(out["resp"].get("content", ""))
    ev = strict_det_evaluate_row(row=row, candidate=candidate, row_no=f"{row['category']}-{row['index']}")
    records.append({
        "cat": row["category"], "idx": row["index"],
        "command": row["command_eng"][:46],
        "premap": snip["retrieval"]["status"],
        "det_score": round(float(ev["det_score"]), 1),
        "det_pass": bool(ev["det_pass"]),
        "gt_exact": bool(ev["gt_exact"]),
        "failure_reasons": ",".join(ev["failure_reasons"])[:44],
    })
    gen_dump.append({"category": row["category"], "index": row["index"],
                     "command_eng": row["command_eng"], "generated": out["parsed"],
                     "raw_content": out["resp"].get("content", ""), "gt": row["gt"],
                     "premap_status": snip["retrieval"]["status"],
                     "premap_shortlist": [h["category"] for h in snip["retrieval"].get("shortlist", [])],
                     "det_score": float(ev["det_score"]), "det_pass": bool(ev["det_pass"]),
                     "failure_reasons": ev["failure_reasons"],
                     "latency_sec": out["latency_sec"]})
    print(f"cat {row['category']} idx {row['index']:>3s} det_score={ev['det_score']:6.1f} "
          f"pass={ev['det_pass']} | {row['command_eng'][:52]}", flush=True)

df = pd.DataFrame(records)
with pd.option_context("display.width", 160, "display.max_columns", None):
    print()
    print(df.to_string(index=False))
print(f"\nsummary: n={len(df)}, mean det_score={df.det_score.mean():.1f}, "
      f"det_pass {int(df.det_pass.sum())}/{len(df)}, gt_exact {int(df.gt_exact.sum())}/{len(df)}")

df.to_csv(ART_DIR / "det_eval_results.csv", index=False)
with (ART_DIR / "generations.jsonl").open("w", encoding="utf-8") as f:
    for rec in gen_dump:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("saved ->", ART_DIR / "det_eval_results.csv", "|", ART_DIR / "generations.jsonl")

cat 1 idx   1 det_score=  92.5 pass=True | Switch the dishwasher to dry mode.


cat 2 idx   1 det_score=  40.5 pass=False | Set the living room light to yellow.


cat 3 idx   1 det_score=  83.0 pass=True | If the audio recorder is recording, stop recording a


cat 4 idx   1 det_score=  92.5 pass=True | When the contact sensor is detected, turn on the lig


cat 5 idx   1 det_score=  92.5 pass=True | Switch the TV to channel 7 and switch to channel 11 


cat 6 idx   1 det_score=  82.1 pass=True | If the temperature is 28 degrees or higher and the h


cat 7 idx   1 det_score=  89.8 pass=True | Every 10 minutes, if the temperature is 30 degrees o


cat 8 idx   1 det_score=  87.9 pass=True | Every hour, repeat opening and closing the door.



cat idx                                        command                            premap  det_score  det_pass  gt_exact                              failure_reasons
  1   1             Switch the dishwasher to dry mode.                              used       92.5      True     False                              period_mismatch
  2   1           Set the living room light to yellow. skipped_connected_devices_present       40.5     False     False period_mismatch,gt_mismatch,gt_service_cover
  3   1 If the audio recorder is recording, stop recor                              used       83.0      True     False period_mismatch,gt_mismatch,gt_service_cover
  4   1 When the contact sensor is detected, turn on t                              used       92.5      True     False                              period_mismatch
  5   1 Switch the TV to channel 7 and switch to chann                              used       92.5      True     False                              period_mismatch
  6   1 I

## 12. (Optional) 서버 엔트리 `gpt_mg.run.generate_joi_code`

실서버(`demo.py`)가 쓰는 엔트리는 `gpt_mg/run.py::generate_joi_code`이며,
`version0_15_update20260413`의 `model_config.json`은 로컬 Qwen2.5-Coder-7B 워커
(backend `local_qwen`)로 설정되어 있습니다. GPU와 로컬 모델이 준비된 환경에서
`RUN_SERVER_ENTRY=true`로 실행해 보세요(로컬 워커 검증 절차는 00/01 노트북 참조).

In [12]:
# ============================================================
# Optional: server entry via gpt_mg.run.generate_joi_code (local Qwen worker)
# ============================================================
if RUN_SERVER_ENTRY:
    sys.path.insert(0, str(BASE_DIR))
    from gpt_mg.run import generate_joi_code
    server_out = generate_joi_code(
        rows280[0]["command_eng"], "gpt_mg.version0_15_update20260413",
        {}, time.strftime("%Y-%m-%d %H:%M:%S"), None)
    print(json.dumps(server_out.get("code", ""), ensure_ascii=False, indent=1)[:1200])
else:
    print("skipped: set RUN_SERVER_ENTRY=true to run the local-Qwen server entry")
    print("(requires the local model under LOCAL_MODEL_BASE and a free GPU; see notebooks 00/01)")

skipped: set RUN_SERVER_ENTRY=true to run the local-Qwen server entry
(requires the local model under LOCAL_MODEL_BASE and a free GPU; see notebooks 00/01)


## 13. Summary

이 노트북이 실행한 논문 파이프라인과 산출물:

| 단계 | 논문 대응 | 산출물 |
|---|---|---|
| 학습쌍 구축 | pre-mapping 학습 데이터 | `finetune_pairs_split.jsonl` |
| e5-small MNRL fine-tune | 채택 검색 구성의 인코더 | `models/e5-small-split-v201/` |
| 보류 쿼리 검색 평가 | 검색 벤치마크 (base vs FT) | 위 셀 출력 |
| Service Pre-mapping | Eq.(1) hybrid w=0.6, split + 절 분해 | 위 셀 shortlist 출력 |
| gpt_mg 검색 번들 | 서버 파이프라인 premap 자산 | `retrieval_bundle_e5_ft/` |
| NL-to-JOILang 생성 | premap → 프롬프트 → gpt-4.1-mini | `prompts/`, `generations.jsonl` |
| strict DET 평가 | 최종 정확도 검증 | `det_eval_results.csv` |

모든 산출물은 `artifacts/premapping_pipeline_runs/<RUN_TAG>/`에 있습니다.

In [13]:
# ============================================================
# Artifact listing
# ============================================================
for p in sorted(ART_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size:>10,d}  {p.relative_to(ART_DIR)}")

       892  det_eval_results.csv
   701,077  finetune_pairs_split.jsonl
     8,864  generations.jsonl
        90  models/e5-small-split-v201/1_Pooling/config.json
    18,174  models/e5-small-split-v201/README.md
       712  models/e5-small-split-v201/config.json
       283  models/e5-small-split-v201/config_sentence_transformers.json
133,462,128  models/e5-small-split-v201/model.safetensors
       429  models/e5-small-split-v201/modules.json
       241  models/e5-small-split-v201/sentence_bert_config.json
   711,649  models/e5-small-split-v201/tokenizer.json
       403  models/e5-small-split-v201/tokenizer_config.json
   111,625  prompts/demo_row1_prompt.txt
    76,928  retrieval_bundle_e5_ft/doc_embs_e5.npy
       237  retrieval_bundle_e5_ft/hybrid_config.json
    12,702  retrieval_bundle_e5_ft/metadata.json
